<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/topic-modelling/Notebook_4_Introduction_to_Topic_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 · What are customers talking about?
**NTU MSc Growth Hacking | Customer Insight Lab **

In Notebook 3, a four-star review could contain a complaint. Sentiment told us how the wording sounded. Now the founder asks **“What was the complaint about?”** Topic modelling helps us look for recurring subjects across reviews.

Work down the cells in order in Colab. Use your existing Oddbox review file. Both models analyse every usable written review in the uploaded file. Save the review files from Notebook 3 for Notebook 5. By the end you will have named topics, compared two methods on one review and tested one decision that could change the findings.

## 1 · Which reviews would you put together?
Upload your existing Oddbox review CSV or Excel workbook. For Excel, select the sheet containing one row per review. Choose the column containing the customer's words. If available, choose the unique review ID. The notebook supplies a row reference for each review.

After the cleaning cell, read the six displayed reviews with a neighbour. Which discuss the same subject? Give each group a short name and point to the words that led you there. The models will use the full cleaned file.

In [ ]:
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
pd.set_option('display.max_colwidth', None)

def choose(options, prompt, default):
    print(prompt)
    for i, label in enumerate(options, 1):
        print(i, label)
    while True:
        entry = input(f'Enter a number or press Return for {default+1}: ').strip()
        if not entry:
            return options[default]
        if entry.isdigit() and 1 <= int(entry) <= len(options):
            return options[int(entry)-1]

uploads = files.upload()
if len(uploads) != 1:
    raise ValueError('Upload one CSV or XLSX file.')
filename, payload = next(iter(uploads.items()))
if filename.lower().endswith('.xlsx'):
    workbook = pd.ExcelFile(io.BytesIO(payload))
    sheet = choose(workbook.sheet_names, 'Select the review sheet', 0)
    frame = workbook.parse(sheet, dtype=str, keep_default_na=False)
elif filename.lower().endswith('.csv'):
    frame = pd.read_csv(io.BytesIO(payload), encoding='utf-8-sig', dtype=str,
                        keep_default_na=False, sep=None, engine='python')
else:
    raise ValueError('Choose a .csv or .xlsx file.')
headers = list(frame.columns)
normal = [re.sub(r'[^a-z0-9]', '', h.lower()) for h in headers]
def suggest(names):
    return next((normal.index(k) for k in names if k in normal), 0)
text_col = choose(headers, 'Which column contains the review wording?',
                  suggest(['reviewtext','reviewbody','text','review','content','message']))
id_col = choose([None] + [h for h in headers if h != text_col], 'Unique review ID (None if unavailable)', 0)
raw = pd.DataFrame({'id': ['R'+str(i) for i in range(1,len(frame)+1)],
                    'source_id': frame[id_col].str.strip() if id_col else '',
                    'review': frame[text_col]})
print(f'{len(raw)} rows imported from {filename}.')


## 2 · What should count as evidence?
A review imported twice gives its words two votes. A blank row adds a customer with no words to analyse. Inspect the rows the cell removes. We keep identical wording from different IDs because two customers could describe the same experience.

The original stays alongside the prepared text. For the first run, we use lower case and remove common words such as “the”. We keep small words such as “not” because they can matter when we return to interpret a review. Removing punctuation and counting words loses clues about tone. Always read the original before naming a topic.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfTransformer
from sklearn.decomposition import LatentDirichletAllocation, NMF
tidy = raw.review.str.strip()
blank = tidy.eq('')
repeat = raw.assign(trimmed=tidy).duplicated(['source_id','trimmed']) & raw.source_id.ne('') & ~blank
display(raw.loc[blank | repeat].assign(reason=np.where(blank[blank | repeat], 'Blank', 'Repeated import')).head(10))
data = raw.loc[~(blank | repeat)].reset_index(drop=True).copy()
print(f'{len(raw)} imported → {len(data)} reviews for analysis')
if len(data) < 3:
    raise ValueError('Use a review column with at least three usable written reviews.')
display(data.sample(min(6,len(data)), random_state=42)[['id','review']])


### Which word forms should count together?
Start with **Keep word forms**. Later, return here to try a different choice.

**Stemming** trims word endings using rules. “Cooking” and “cooked” can both become “cook”, and some stems look unfamiliar.

**Lemmatisation** uses a dictionary and the word’s grammatical role to find a base form. For instance, “pears” becomes “pear” as a noun and “cooking” can become “cook” as a verb.

Compare the original and prepared versions below. If you merge forms, which useful links could become easier to spot? Which distinctions might disappear?

In [ ]:
word_form = "Lemmatisation" #@param ["Keep word forms", "Stemming", "Lemmatisation"]
stop = set(ENGLISH_STOP_WORDS) - {'no','not','never','oddbox','box'}
if word_form == 'Stemming':
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
elif word_form == 'Lemmatisation':
    import nltk
    from nltk.stem import WordNetLemmatizer
    nltk.download('wordnet', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    lemma = WordNetLemmatizer()

def prepare(text):
    words = re.findall(r"[a-z]+", text.lower())
    if word_form == 'Lemmatisation':
        pos_map = {'N':'n', 'V':'v', 'J':'a', 'R':'r'}
        return ' '.join(lemma.lemmatize(w, pos_map.get(tag[0], 'n'))
                        for w, tag in nltk.pos_tag(words) if w not in stop)
    words = [w for w in words if w not in stop]
    if word_form == 'Stemming':
        words = [stemmer.stem(w) for w in words]
    return ' '.join(words)

data['prepared'] = data.review.map(prepare)
display(data[['id','review','prepared']].head(6))

## 3 · How does a review become a row of numbers?
Each row represents one review. Each column represents a word. A count of 1 under “driver” means the word appeared once in that review. Look at the first two rows. Which columns would help you tell their subjects apart?

**LDA** will use these counts. **NMF** will use a weighted version called **TF–IDF**. It gives relatively more weight to words that distinguish one review from others. For example, if every review mentioned “box”, that word alone would do little to separate delivery comments from cooking comments. Compare numbers within each table. The tables use different scales.

We also count pairs of neighbouring words in the prepared text. Removing common words can bring two words together, so check the original review before quoting a phrase.

In [ ]:
vectoriser = CountVectorizer(ngram_range=(1, 2))
usable_words = data.prepared.str.contains(r'\b\w\w+\b')
print(f'{(~usable_words).sum()} reviews have no usable terms after preparation.')
model_data = data.loc[usable_words].reset_index(drop=True)
if len(model_data) < 2:
    raise ValueError('Too few reviews have usable words. Check your text column.')
counts = vectoriser.fit_transform(model_data.prepared)
terms = vectoriser.get_feature_names_out()
weighted = TfidfTransformer().fit_transform(counts)
# Show words actually found in the first two reviews.
visible = np.flatnonzero(counts[:2].sum(axis=0).A1)[:12]
for title, matrix in [('Word counts', counts), ('TF–IDF weights', weighted)]:
    print(title)
    display(pd.DataFrame(matrix[:2, visible].toarray(), columns=terms[visible],
                         index=model_data.id[:2]).round(2))

## 4 · Can the models recover your groups?
Start with three topics. **LDA** looks for words that tend to occur together. It treats each topic as a distribution over words and each review as a mixture of topics. A delivery review could also mention bruised fruit.

**NMF** looks for a small set of word patterns that, added together in different amounts, approximate the weighted table. A review might need a lot of the delivery pattern and a little of the fruit pattern to account for its wording.

Neither method supplies a name. Topic numbers are local to each model: LDA topic 1 and NMF topic 1 can refer to different subjects. Match topics by reading their words and reviews.

In [ ]:
number_of_topics = 3 #@param [2,3,4] {type:"raw"}
if number_of_topics > min(counts.shape):
    raise ValueError('Choose fewer topics or upload more reviews with distinct words.')
models = {
    'LDA': LatentDirichletAllocation(n_components=number_of_topics, random_state=42, max_iter=40),
    'NMF': NMF(n_components=number_of_topics, init='nndsvda', random_state=42, max_iter=800)
}
shares = {}
for method, model in models.items():
    strength = model.fit_transform(counts if method == 'LDA' else weighted)
    totals = strength.sum(axis=1, keepdims=True)
    shares[method] = np.divide(strength, totals, out=np.zeros_like(strength), where=totals>0)

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.transforms import Bbox

def topic_page(method, i, label):
    weights = models[method].components_[i]
    order = np.argsort(-weights, kind='stable')
    top = [j for j in order if weights[j] > 0][:12]
    fig = plt.figure(figsize=(11.7, 8.3))
    fig.suptitle(f'{method} topic {i+1}  |  {label}', fontsize=16, wrap=True)
    cloud = fig.add_axes([.025, .20, .52, .65], xlim=(0,1), ylim=(0,1))
    cloud.axis('off')
    table = fig.add_axes([.56, .16, .42, .69])
    table.axis('off')
    table.set_title('Ranked words and two-word phrases', loc='left')
    renderer = fig.canvas.get_renderer()
    occupied = []
    ranked = []
    for rank, j in enumerate(top, 1):
        term = terms[j]
        share = 100 * weights[j] / weights.sum()
        ranked.append([rank, term, f'{share:.2f}%'])
        size = 12 + 27 * np.sqrt(weights[j] / weights[top[0]])
        artist = cloud.text(.5,.5,term,ha='center',va='center',fontsize=size,
                            color=plt.cm.viridis(.18+.55*(rank-1)/12))
        for attempt in range(1500):
            radius = .013 * np.sqrt(attempt)
            angle = attempt * 2.4
            artist.set_position((.5+radius*np.cos(angle),.5+radius*np.sin(angle)))
            box = artist.get_window_extent(renderer).expanded(1.04,1.08)
            if cloud.bbox.contains(*box.min) and cloud.bbox.contains(*box.max) and not any(box.overlaps(b) for b in occupied):
                occupied.append(box)
                break
            if attempt in (500,1000):
                artist.set_fontsize(artist.get_fontsize()*.78)
        else:
            artist.remove()
    if ranked:
        table.table(cellText=ranked,colLabels=['Rank','Term','Topic weight'],cellLoc='left',
                    colWidths=[.13,.61,.26],bbox=[0,.04,1,.92])
    fig.text(.04,.10,'Topic weight = this term’s share of the total weight across every term in this topic.')
    fig.text(.04,.06,'Only the leading terms are shown. Compare these with the wording of the original reviews.')
    return fig

for method in models:
    for i in range(number_of_topics):
        fig = topic_page(method,i,'What would you call this topic?')
        plt.show()
        plt.close(fig)
        best = int(shares[method][:,i].argmax())
        print(f'Example {model_data.id[best]} | {model_data.review[best]}')


**Name each topic.** Read the numbered list alongside each cloud. A term with 4% accounts for four hundredths of this topic’s total term weight. This differs from the percentage of a review assigned to a topic. Pick two high-ranking terms and explain how the example review links them. Then read its example review. “Doorstep delivery” is a more useful name than “negative” if the evidence is about where the driver left the box. Enter your own names when prompted. If the same subject appears in both models, use the same name so you can compare them.

In [9]:
names = {}
for method in models:
    names[method] = []
    for i in range(number_of_topics):
        default = f'{method} topic {i+1}'
        names[method].append(input(f'Name {default}: ').strip() or default)

Name LDA topic 1: delivery
Name LDA topic 2: qulaity
Name LDA topic 3: service
Name NMF topic 1: delveiry
Name NMF topic 2: quality
Name NMF topic 3: service


## 5 · Can one review mention two topics?
Choose a review from the preview below. Read it and predict which subjects you think it contains. Which topic would you choose if you were allowed only one?

For LDA, read the percentages as estimated topic shares. For NMF, we scale each review’s weights to total 100% so you can see its relative fit to each pattern. NMF percentages are descriptive shares of weights. Neither column tells you the probability that your chosen topic name is correct.

In [ ]:
display(model_data[['id','review']].sample(min(6,len(model_data)), random_state=42))
review_id = input('Enter a displayed review ID, or press Return for the first review in the file: ').strip() or model_data.id.iloc[0]
matches = model_data.index[model_data.id.eq(review_id.strip().upper())]
if len(matches) != 1:
    raise ValueError('Choose an ID from the displayed reviews.')
row = matches[0]
print(model_data.review[row])
output = model_data.copy()
for method in models:
    weights = shares[method]
    for i, name in enumerate(names[method]):
        output[f'{method} {i+1} {name} (%)'] = (100*weights[:,i]).round(1)
    output[method+' main topic'] = [names[method][v.argmax()] if v.sum() else 'No matching words' for v in weights]
    table = pd.DataFrame({'topic':names[method], 'share (%)':(100*weights[row]).round(1)})
    print(method)
    display(table.sort_values('share (%)',ascending=False))
display(output[['id','review'] + [col for col in output if col.endswith('(%)')]].head(12))

**Discuss with a neighbour.** Did both models pick up the subjects you identified in your chosen review? Quote one phrase that supports the main topic and one that the main label overlooks. If the models disagree, explain which interpretation you find more useful for the founder.

## 6 · Change one decision
Write down your current finding for your chosen review. Now either change the topic count from three to two in Block 4, **or** change the word-form setting in Block 2. Rerun from the cell you changed through Block 5, including the naming cell. Keep everything else fixed.

**Record your comparison:** I changed ___. Before, review ___ was mainly about ___. Afterwards, ___. The words ___ support my preferred reading. Which detail would the founder lose if we reported only the main topic?

Download the most recent run below. Notebook 5 will repeat the comparison on the positive and negative review files you selected in Notebook 3. Those files let you investigate whether the same subject appears in both groups.

The ZIP contains the labelled cloud PDF and a CSV with every analysed review’s percentage for each LDA and NMF topic. The PDF ranks terms within a topic. The CSV shows how each review is split across topics.

In [12]:
from google.colab import files
output['word_form'] = word_form
output['number_of_topics'] = number_of_topics
output['input_file'] = filename
filename_export = 'Notebook-4-review-topics.csv'
output.to_csv(filename_export, index=False, encoding='utf-8-sig')
pdf_name = 'Notebook-4-topic-clouds.pdf'
with PdfPages(pdf_name) as pdf:
    for method in models:
        for i, label in enumerate(names[method]):
            fig = topic_page(method, i, label)
            pdf.savefig(fig)
            plt.close(fig)
import zipfile
download = 'Notebook-4-topic-results.zip'
with zipfile.ZipFile(download, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pdf_name)
    bundle.write(filename_export)
files.download(download)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>